In [2]:
%pwd

'/ceph/MethDev/JW240627--at-snmCT_with_TE/mCT_with_TE/kay'

In [1]:
import os, re, glob
import numpy as np
import pandas as pd

# ---------- CONFIG ----------
# Your input pattern (as you provided)
IN_GFF_GLOB = "/ceph/MethDev/JW240627--at-snmCT_with_TE/mCT_with_TE/filtered_w100_gffs_by_clst_genotype/filtered_CG_w100_gffs_by_clst_genotype_ct/clst*.col.CG.w100.gff"
OUT_DIR = "/ceph/MethDev/pbio/kay/data/bigwig_out"  # change if you like
TRACK_LABEL_PREFIX = "mCG"  # used only for naming files

# Map numeric chr -> RefSeq names that IGV shows in your screenshot
# (adjust versions if your IGV title bar shows different ones)
REFSEQ_MAP = {
    "1": "NC_003070.9",
    "2": "NC_003071.7",
    "3": "NC_003074.8",
    "4": "NC_003075.7",
    "5": "NC_003076.8",
}

# TAIR10-ish sizes (exact for the five nuclear chromosomes)
CHROM_SIZES = {
    "NC_003070.9": 30427671,
    "NC_003071.7": 19698289,
    "NC_003074.8": 23459830,
    "NC_003075.7": 18585056,
    "NC_003076.8": 26975502
}

# ---------- HELPERS ----------
def _parse_meth_score(cols):
    """
    Return meth_score (float) from a GFF row represented as a Series with fields:
    seqid, start, end, score, attributes.
    Accepts meth_score in:
      - the numeric 'score' column (col 6), or
      - attributes like 'meth_score=0.123' or 'score=0.123' or 'frac=0.123'
    """
    # 1) GFF score column if numeric
    sc = cols.get("score", None)
    if sc not in (None, ".", ""):
        try:
            return float(sc)
        except ValueError:
            pass

    # 2) Look in attributes key-value pairs
    attr = str(cols.get("attributes", ""))
    # common keys: meth_score, m, frac, score
    m = re.search(r'(?:meth_score|m|frac|score)\s*=\s*([0-9]*\.?[0-9]+(?:[eE][-+]?\d+)?)', attr)
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return np.nan
    return np.nan

def _map_seqid_to_refseq(seqid: str) -> str:
    """
    Map incoming seqid ('1','chr1','Chr1','NC_003070.9', etc.) to RefSeq names IGV uses.
    Leaves it alone if it already looks like 'NC_00...'.
    """
    s = str(seqid)
    if s.startswith("NC_"):
        return s
    # normalize common forms
    s_norm = s.replace("Chr", "chr")
    if s_norm.startswith("chr"):
        s_norm = s_norm.replace("chr", "")
    # now expect '1'..'5'
    return REFSEQ_MAP.get(s_norm, s)  # fall back to original if unknown

def gff_to_bigwig(gff_path: str, out_bw_path: str):
    """
    Convert one GFF → bigWig using pyBigWig.
    Assumes windows are non-overlapping 1-based closed intervals with a meth score per row.
    """
    import pyBigWig

    # Read minimal GFF columns
    names = ["seqid","source","type","start","end","score","strand","phase","attributes"]
    usecols = [0,2,3,4,5,8]  # seqid,type,start,end,score,attributes
    df = pd.read_csv(
        gff_path, sep="\t", comment="#", header=None, names=names, usecols=usecols,
        dtype={0:str,2:str,3:int,4:int,5:str,8:str}, engine="c", na_filter=False
    )
    if df.empty:
        raise ValueError(f"{gff_path}: no data rows found")

    # Extract meth_score per row
    df["meth_score"] = df.apply(_parse_meth_score, axis=1).astype(float)
    # Basic cleaning
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["meth_score"])
    # clip scores to [0,1] (adjust if your scale is 0..100)
    df["meth_score"] = df["meth_score"].clip(0, 1)

    # Map chromosomes to RefSeq
    df["chrom"] = df["seqid"].map(_map_seqid_to_refseq)

    # Convert to 0-based, half-open
    df["start0"] = df["start"].astype(int) - 1
    df["end1"]   = df["end"].astype(int)

    # Keep only rows where chrom is known and coordinates are sane
    df = df[(df["start0"] >= 0) & (df["end1"] > df["start0"]) & df["chrom"].notna()].copy()

    # Sort by chrom, start (required by bigWig)
    df = df.sort_values(["chrom","start0","end1"], kind="mergesort").reset_index(drop=True)

    # Build header: use known sizes when available, otherwise infer minimal size from data
    chroms_present = df["chrom"].unique().tolist()
    header = {}
    for c in chroms_present:
        if c in CHROM_SIZES:
            header[c] = CHROM_SIZES[c]
        else:
            # Infer size as max end seen (safe for subset tracks)
            header[c] = int(df.loc[df["chrom"]==c, "end1"].max())

    # Write bigWig
    os.makedirs(os.path.dirname(out_bw_path), exist_ok=True)
    bw = pyBigWig.open(out_bw_path, "w")
    bw.addHeader(list(header.items()))

    # Add entries per chromosome (sorted)
    for c, g in df.groupby("chrom", sort=False):
        bw.addEntries(
            [c] * len(g),
            g["start0"].astype(int).tolist(),
            ends=g["end1"].astype(int).tolist(),
            values=g["meth_score"].astype(float).tolist()
        )
    bw.close()
    return out_bw_path

# ---------- RUN ----------
os.makedirs(OUT_DIR, exist_ok=True)
gffs = sorted(glob.glob(IN_GFF_GLOB))
if not gffs:
    raise SystemExit(f"No files matched: {IN_GFF_GLOB}")

print(f"Found {len(gffs)} GFF files.")
made = []
for gff in gffs:
    base = os.path.basename(gff)
    name = os.path.splitext(base)[0]  # e.g., 'clstX.col.CG.w100'
    out_bw = os.path.join(OUT_DIR, f"{TRACK_LABEL_PREFIX}_{name}.bw")
    print(f"→ {out_bw}")
    gff_to_bigwig(gff, out_bw)
    made.append(out_bw)

print(f"Done. Wrote {len(made)} bigWigs to: {OUT_DIR}")


Found 17 GFF files.
→ ./data/bigwig_out/mCG_clst0.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst1.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst10.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst11.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst12.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst13.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst14.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst15.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst16.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst2.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst3.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst4.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst5.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst6.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst7.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst8.col.CG.w100.bw
→ ./data/bigwig_out/mCG_clst9.col.CG.w100.bw
Done. Wrote 17 bigWigs to: ./data/bigwig_out
